# Cantilever Beam Validation Walkthrough

This notebook demonstrates the smodal workflow against an analytical reference with known answers.
It is intended as a first-run guide: follow the steps in sequence, compare your results to the tables
below, and use the discrepancies (or their absence) to build intuition about each page.

---

## Beam Configuration

```
Wall (clamped)                                          Tip mass M
│▓▓▓▓▓▓▓▓▓▓▓▓▓════════════════════════════════════════[ M ]
x = 0 m        acc_0m   acc_5m    acc_7m            acc_10m
                   ↑       ↑          ↑                  ↑
                (=0)    sensor      sensor             sensor + force
```

| Parameter | Value |
|-----------|-------|
| Material | Steel (E = 210 GPa, ρ = 7850 kg/m³) |
| Cross-section | Circular tube, OD = 0.1 m, wall = 0.01 m |
| Length | L = 10 m |
| Tip mass | M = 100 kg at x = 10 m |
| Boundary condition | Clamped at x = 0 |
| Excitation | Random force at tip (x = 10 m) |
| Sensors | 4 accelerometers at x = 0, 5, 7, 10 m (Z-axis) |
| Damping (assigned) | ξ = 2 % for all modes |

**Key note — sensor at x = 0 m:** The beam is clamped at x = 0. An accelerometer at the wall reads
zero displacement for every mode (this is the clamped boundary condition). Column `acc_0m` in the CSV
is therefore identically zero and carries no modal information. It is included to demonstrate how
the app handles zero-motion channels.

---

## Analytical Reference — Natural Frequencies

Solved from the characteristic equation for a cantilever with tip mass:

$$1 + \cos(\beta L)\cosh(\beta L) + \varepsilon (\beta L)[\sinh(\beta L)\cos(\beta L) - \cosh(\beta L)\sin(\beta L)] = 0$$

where $\varepsilon = M / (\rho A L) = 0.4505$.

| Mode | $\beta L$ | $f_n$ (Hz) | $\xi$ (%) |
|------|-----------|------------|----------|
| 1 | 1.4447 | **0.550** | 2.0 |
| 2 | 4.1262 | **4.487** | 2.0 |
| 3 | 7.2016 | **13.668** | 2.0 |
| 4 | 10.307 | **27.998** | 2.0 |

In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = pathlib.Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CSV_PATH = ROOT / "data" / "input" / "cantilever_beam" / "cantilever_response.csv"
BDF_PATH = ROOT / "data" / "input" / "cantilever_beam" / "cantilever_wireframe.bdf"
F06_PATH = ROOT / "data" / "input" / "cantilever_beam" / "cantilever_modes.f06"

assert CSV_PATH.exists(), f"Run scripts/generate_cantilever_reference.py first\n{CSV_PATH}"
print("Data files found.")

---
## Step 1 — Inspect the Time History

The CSV has six columns: `time`, `force`, `acc_0m`, `acc_5m`, `acc_7m`, `acc_10m`.
Duration is 300 s at fs = 200 Hz (60 000 rows).

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Rows: {len(df)}, Columns: {list(df.columns)}")
print(f"Duration: {df['time'].iloc[-1]:.1f} s  |  Sample rate: {1/(df['time'].iloc[1]-df['time'].iloc[0]):.0f} Hz")
print()
print(df.describe().round(4))

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(12, 9), sharex=True)
cols = ["force", "acc_0m", "acc_5m", "acc_7m", "acc_10m"]
labels = ["Force (N)", "acc_0m (m/s²)\n[clamped: =0]", "acc_5m (m/s²)", "acc_7m (m/s²)", "acc_10m (m/s²)"]
t = df["time"].values
for ax, col, label in zip(axes, cols, labels):
    ax.plot(t[:2000], df[col].values[:2000], lw=0.6)  # first 10 s
    ax.set_ylabel(label, fontsize=8)
    ax.grid(alpha=0.3)
axes[-1].set_xlabel("Time (s)")
fig.suptitle("Cantilever beam — first 10 s of time history")
plt.tight_layout()
plt.show()

### App — Page 1 (Time History)

1. Start the app: `streamlit run app.py`
2. Enter an analysis name (e.g., `cantilever_tutorial`) and click **Save**.
3. On **Page 1**, upload `cantilever_response.csv`.
4. Set **Input channel** = `force`, **Output channels** = `acc_5m, acc_7m, acc_10m`.
   (Leave `acc_0m` unselected — it is the clamped wall and carries no modal information.)
5. Keep the full time range (no trim). Click **Save analysis log**.

---
## Step 2 — Spectral Analysis and FRF

Here we compute the H1 FRF estimate directly in Python to understand what the app will show.

In [ ]:
from core.spectral import compute_welch_quantities

fs = 200.0
nperseg = 8192      # 40.96 s segments → df ≈ 0.024 Hz (needed for mode 1 at 0.55 Hz)
noverlap = 4096

force = df["force"].values
sensors = {"acc_5m": df["acc_5m"].values,
           "acc_7m": df["acc_7m"].values,
           "acc_10m": df["acc_10m"].values}

frfs = {}
for name, acc in sensors.items():
    result = compute_welch_quantities(force, acc, fs, nperseg=nperseg, noverlap=noverlap)
    frfs[name] = result

freqs = frfs["acc_5m"]["freqs"]
print(f"FRF frequency grid: {len(freqs)} bins, df = {freqs[1]-freqs[0]:.4f} Hz, max = {freqs[-1]:.1f} Hz")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

for name, res in frfs.items():
    H1_db = 20 * np.log10(np.abs(res["H1"]) + 1e-30)
    ax1.plot(res["freqs"], H1_db, label=name, lw=0.8)

ax1.set_ylabel("|H1| (dB)")
ax1.set_xlim(0, 40)
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

# Mark analytical frequencies
expected_fn = [0.550, 4.487, 13.668, 27.998]
for i, fn in enumerate(expected_fn):
    ax1.axvline(fn, color="red", lw=0.7, ls="--", alpha=0.6,
                label=f"Mode {i+1}: {fn:.3f} Hz" if i == 0 else None)
    ax1.text(fn + 0.1, ax1.get_ylim()[0] + 5, f"M{i+1}\n{fn:.2f}", fontsize=7, color="red")

# Coherence
for name, res in frfs.items():
    ax2.plot(res["freqs"], res["gamma2"], label=name, lw=0.8)
ax2.axhline(0.7, color="orange", ls="--", lw=1, label="γ² = 0.7")
ax2.set_xlabel("Frequency (Hz)")
ax2.set_ylabel("Coherence γ²")
ax2.set_ylim(-0.05, 1.1)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

fig.suptitle("H1 FRF and coherence — cantilever beam")
plt.tight_layout()
plt.show()

print("Red dashed lines mark analytical frequencies.")
print("Mode 1 at 0.55 Hz is very close to DC — you may need to zoom in.")

### App — Page 3 (Spectral Analysis)

1. Navigate to **Page 3** in the app.
2. Set **nperseg = 8192** (needed to resolve mode 1 at 0.55 Hz; the default 2048 is too coarse).
3. Inspect the **FRF** tab: you should see four peaks at ≈ 0.55, 4.49, 13.67, 28.0 Hz.
4. Inspect the **Coherence** tab: values should be close to 1.0 at each resonance.

**Why nperseg = 8192?** Mode 1 has a half-power bandwidth of only 0.022 Hz  
(2 × ξ × fn = 2 × 0.02 × 0.55). With nperseg = 2048 (df ≈ 0.098 Hz) the peak is spread  
over just one frequency bin and appears artificially damped. nperseg = 8192 gives df ≈ 0.024 Hz,  
resolving the peak across ~2 bins.

---
## Step 3 — SIMO System Identification

Use pLSCF to extract the four modes from the FRF estimated above.

In [ ]:
from core.sysid import build_stability_table, compute_cmif, deduplicate_stable_poles

# Stack H1 from all three sensors into (n_freqs, n_out)
H1_stack = np.column_stack([frfs[name]["H1"] for name in ["acc_5m", "acc_7m", "acc_10m"]])

# Restrict to 0–35 Hz (below Nyquist, above noise floor)
mask = (freqs >= 0.1) & (freqs <= 35.0)
H_band = H1_stack[mask]
f_band = freqs[mask]

stab = build_stability_table(H_band, f_band, fs=fs, max_order=16)
deduped = deduplicate_stable_poles(stab)

print(f"{'Mode':>5}  {'fn_est (Hz)':>12}  {'xi_est (%)':>10}  {'status':>10}")
print("-" * 45)
for d in sorted(deduped, key=lambda x: x["fn_hz"]):
    print(f"       {d['fn_hz']:12.4f}  {d['xi_pct']:10.2f}  {'stable_all':>10}")

In [ ]:
# Compare against analytical reference
expected = [(0.550, 2.0), (4.487, 2.0), (13.668, 2.0), (27.998, 2.0)]
recovered = sorted([(d["fn_hz"], d["xi_pct"]) for d in deduped])

print(f"{'Mode':>4}  {'fn_analytical':>14}  {'fn_estimated':>14}  {'fn_error_%':>12}  {'xi_analytical':>14}  {'xi_estimated':>14}")
print("-" * 80)
for i, (fn_a, xi_a) in enumerate(expected):
    if i < len(recovered):
        fn_e, xi_e = recovered[i]
        fn_err = abs(fn_e - fn_a) / fn_a * 100
        xi_err = abs(xi_e - xi_a) / xi_a * 100
        print(f"{i+1:>4}  {fn_a:>14.4f}  {fn_e:>14.4f}  {fn_err:>12.3f}  {xi_a:>14.1f}  {xi_e:>14.2f}")
    else:
        print(f"{i+1:>4}  {fn_a:>14.4f}  {'— not found':>14}")

**Expected results:**
- Frequency error < 1 % for all four modes.
- Damping error ≤ ±15 % relative (i.e., xi ≈ 2.0 ± 0.3 %).
- Mode 1 may have larger error due to limited frequency resolution at 0.55 Hz.

### App — Page 4 (SIMO)

1. Navigate to **Page 4**.
2. Set **Input channel** = `force`, **Output channels** = `acc_5m, acc_7m, acc_10m`.
3. Set frequency band: **0.1 – 35 Hz**. Set max model order to **16**.
4. On the stability diagram, look for four vertical clusters of stable markers.
5. Click each cluster to extract the four modes. Check the NMSE fits.

---
## Step 4 — MAC Comparison Against Analytical Modes

Now load the analytical F06 file to compare estimated vs reference mode shapes.

In [ ]:
from core.geometry import parse_f06

ref = parse_f06(str(F06_PATH))
print("F06 frequencies (Hz):", ref["frequencies_hz"].round(4))
print(f"Mode shapes: {len(ref['mode_shapes'])} modes × {len(ref['mode_shapes'][0])} GIDs")

In [ ]:
# Plot analytical mode shapes along the beam
x_nodes = list(range(11))  # GIDs 1–11 at x = 0..10 m
fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=False)

for k, (ax, ms) in enumerate(zip(axes, ref["mode_shapes"])):
    t3_vals = [ms[gid][2] for gid in range(1, 12)]  # T3 = Z-displacement
    ax.plot(x_nodes, t3_vals, "o-", ms=5)
    ax.axhline(0, color="gray", lw=0.5, ls="--")
    ax.axvline(0, color="red", lw=1.5, ls="-", alpha=0.5, label="Clamp")
    ax.set_title(f"Mode {k+1}\nfn = {ref['frequencies_hz'][k]:.3f} Hz", fontsize=9)
    ax.set_xlabel("x (m)")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Mass-normalized displacement")
axes[0].legend(fontsize=8)
fig.suptitle("Analytical mode shapes — cantilever beam with tip mass")
plt.tight_layout()
plt.show()

print("Note: Mode 1 has no node (monotonic shape).")
print("Mode 2 has one internal node; Mode 3 has two; Mode 4 has three.")
print("GID 1 (x=0, clamped) is always zero — this appears as the clamp boundary.")

### App — Page 7 (MAC)

1. Navigate to **Page 7** in the app.
2. Under **FE reference**, upload `cantilever_modes.f06`.
3. Map the experimental channels to GIDs:  
   `acc_5m` → GID 6, `acc_7m` → GID 8, `acc_10m` → GID 11.
4. Run the MAC computation.

**Expected results:** The MAC matrix diagonal should be ≥ 0.9 for modes 2–4.  
Mode 1 MAC may be lower due to the limited DOF count (only 3 non-zero sensors) and  
the mode shape being nearly linear (difficult to distinguish from a rigid-body motion  
with only 3 measurement points).

**GID 1 (x=0m) is excluded:** The clamped wall has zero displacement, so `acc_0m = 0`.
Including a zero-displacement channel in the MAC produces a degenerate column with no  
discriminating information and should be omitted from the mapping.

---
## Step 5 — Wireframe Visualisation

Load the BDF geometry file to visualise mode shapes on the beam wireframe.

In [ ]:
from core.geometry import parse_wireframe_bdf

geom = parse_wireframe_bdf(str(BDF_PATH))
print(f"Wireframe: {len(geom.grids)} GRIDs, {len(geom.plotels)} PLOTELs")
print("GRIDs at x positions:", sorted([g.x for g in geom.grids.values()]))

### App — Page 8 (Wireframe)

1. Navigate to **Page 8** in the app.
2. Upload `cantilever_wireframe.bdf`.
3. Map experimental mode shape channels to GIDs (same as in Page 7).
4. Select a mode and animate — you should see the beam bending shape.

**Note:** The wireframe has GIDs 1–11 along the beam spine only (no cross-section rings or
side rails). The mode shape animation will show the spine deflecting.
GID 1 at x = 0 m stays at zero for all modes (clamped boundary).

---
## Summary

| Step | Expected result | Troubleshooting |
|------|-----------------|-----------------|
| FRF peaks | 4 peaks at 0.55, 4.49, 13.67, 28.0 Hz | Use nperseg ≥ 8192 to resolve mode 1 |
| Coherence | γ² ≥ 0.9 at resonances | Low coherence → increase record length |
| Stability diagram | 4 stable-all clusters | Set max order ≥ 12 |
| fn error | < 1 % for modes 2–4 | Mode 1 may be 2–5 % due to Welch resolution |
| ξ error | ≤ 15 % for modes 2–4 | Mode 1 damping less reliable |
| MAC diagonal | ≥ 0.9 for modes 2–4 | Only 3 non-zero DOFs — modes 1 may be ambiguous |

### Why is mode 1 hardest to identify?

- **Very low frequency (0.55 Hz):** The Welch estimator needs nperseg ≥ 8192 to resolve the narrow  
  resonance (half-power bandwidth = 0.022 Hz).
- **Near-linear mode shape:** With only 3 non-zero sensors, mode 1's monotonic deflection is  
  nearly indistinguishable from a rigid-body ramp. Adding a sensor at x = 2 m or x = 3 m would help.
- **Near DC:** Very-low-frequency modes are affected by DC offset and drift in accelerometer data.

These are real-world challenges, not simulation artifacts — the analytical reference exposes them
clearly because you know the ground truth.